In [14]:
# =============================================================================
# Model 2 (Retention Intelligence LLM) — Train BOTH sizes, evaluate on your
# held-out test.jsonl, keep both merged models, export GGUF for both, and
# clearly flag the winner. Runs on a free Colab T4 GPU.
# =============================================================================
# Runtime > Change runtime type > T4 GPU, then run cells top to bottom.
# Upload train.jsonl, val.jsonl, test.jsonl to the Colab session first
# (or mount Google Drive and point the paths at your Drive folder).
# =============================================================================

# --- Cell 1: install deps ---------------------------------------------------
!pip install -q transformers peft bitsandbytes trl accelerate datasets



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.7 MB/s eta 0:00:00


In [16]:
import gc
import json
import os
import random
import time

import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [13]:
random.seed(42)

TRAIN_PATH = "model2_train.jsonl"
VAL_PATH = "model2_validation.jsonl"
TEST_PATH = "model2_test.jsonl"

MODEL_CANDIDATES = [
    {"name": "Qwen/Qwen2.5-0.5B-Instruct", "tag": "0.5b", "out_dir": "model2-0.5b-lora"},
    {"name": "Qwen/Qwen2.5-1.5B-Instruct", "tag": "1.5b", "out_dir": "model2-1.5b-lora"},
]

ALLOWED_PREFIXES = {"rm_call", "rate_offer", "fee_waiver", "complaint_escalation", "do_nothing"}

In [14]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f]


train_records = load_jsonl(TRAIN_PATH)
val_records = load_jsonl(VAL_PATH)
test_records = load_jsonl(TEST_PATH)
print(f"Train: {len(train_records)}  Val: {len(val_records)}  Test: {len(test_records)}")

train_ds = Dataset.from_list(train_records)
val_ds = Dataset.from_list(val_records)

Train: 802  Val: 99  Test: 99


In [19]:
# --- Cell 2: training function (reused per model) ---------------------------
def train_one_model(model_name, out_dir):
    print(f"\n{'='*70}\nTraining {model_name}\n{'='*70}")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    def format_example(example):
        text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
        return {"text": text}

    train_fmt = train_ds.map(format_example)
    val_fmt = val_ds.map(format_example)

    sft_config = SFTConfig(
        output_dir=out_dir,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        bf16=True,
        max_length=1024, # Changed from max_seq_length to max_length
        dataset_text_field="text",
        report_to="none",
    )

    trainer = SFTTrainer(model=model, args=sft_config, train_dataset=train_fmt, eval_dataset=val_fmt)
    trainer.train()
    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)

    return model, tokenizer

In [20]:
# --- Cell 3: evaluation function (reused per model) --------------------------
def extract_json_numbers(obj, acc=None):
    if acc is None:
        acc = set()
    if isinstance(obj, dict):
        for v in obj.values():
            extract_json_numbers(v, acc)
    elif isinstance(obj, list):
        for v in obj:
            extract_json_numbers(v, acc)
    elif isinstance(obj, (int, float)):
        acc.add(abs(round(obj)))
    return acc


def extract_text_numbers(text_list):
    import re
    nums = set()
    for t in text_list:
        for m in re.findall(r"-?\d+\.?\d*", t):
            try:
                nums.add(abs(round(float(m))))
            except ValueError:
                pass
    return nums

In [21]:
def evaluate_model(model, tokenizer, test_records, max_new_tokens=300):
    model.eval()
    n = len(test_records)
    json_valid = 0
    prefixes_valid = 0
    grounded_checked = 0
    grounded_ok = 0
    latencies = []

    for rec in test_records:
        system_msg, user_msg, _ = rec["messages"]
        prompt = tokenizer.apply_chat_template(
            [system_msg, user_msg], tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        t0 = time.time()
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens, temperature=0.3, do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
            )
        latencies.append(time.time() - t0)

        completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

        try:
            gen = json.loads(completion)
            assert "why" in gen and "next_actions" in gen and isinstance(gen["why"], list) and isinstance(gen["next_actions"], list)
            json_valid += 1
        except Exception:
            continue  # can't score prefix/grounding on invalid JSON

        prefixes = [a.split(":", 1)[0].strip() for a in gen["next_actions"] if ":" in a]
        if prefixes and all(p in ALLOWED_PREFIXES for p in prefixes):
            prefixes_valid += 1

        user_obj = json.loads(user_msg["content"])
        json_nums = extract_json_numbers(user_obj)
        why_nums = extract_text_numbers(gen["why"])
        if why_nums:
            grounded_checked += 1
            if why_nums & json_nums:
                grounded_ok += 1

    json_valid_rate = json_valid / n
    prefix_valid_rate = prefixes_valid / n
    grounding_rate = (grounded_ok / grounded_checked) if grounded_checked else None
    avg_latency = sum(latencies) / len(latencies)

    # composite: JSON validity and correct action vocabulary matter most for
    # a production dashboard integration; grounding is a secondary quality signal.
    composite = 0.5 * json_valid_rate + 0.3 * prefix_valid_rate + 0.2 * (grounding_rate or 0)

    return {
        "json_valid_rate": round(json_valid_rate, 3),
        "prefix_valid_rate": round(prefix_valid_rate, 3),
        "grounding_rate": round(grounding_rate, 3) if grounding_rate is not None else None,
        "avg_latency_sec": round(avg_latency, 2),
        "composite_score": round(composite, 3),
    }

In [22]:
# --- Cell 4: run both models end-to-end ------------------------------------
results = {}
trained_models = {}  # keep in memory only long enough to eval + merge, then free

for cfg in MODEL_CANDIDATES:
    model, tokenizer = train_one_model(cfg["name"], cfg["out_dir"])
    metrics = evaluate_model(model, tokenizer, test_records)
    metrics["out_dir"] = cfg["out_dir"]
    metrics["base_model"] = cfg["name"]
    results[cfg["tag"]] = metrics
    print(f"\n{cfg['tag']} results: {json.dumps(metrics, indent=2)}")

    # free GPU memory before loading the next candidate
    del model
    gc.collect()
    torch.cuda.empty_cache()


Training Qwen/Qwen2.5-0.5B-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


Map:   0%|          | 0/802 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/99 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.271487,0.258707,0.269549,532775.000000,0.909776


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.271487,0.258707,0.269549,532775.000000,0.909776
2,0.222983,0.226307,0.228117,1065550.000000,0.918828


KeyboardInterrupt: 

In [10]:
# !zip -r /content/my_archive.zip /content/model2-0.5b-lora/ /content/model2-1.5b-lora/
!unzip my_archive.zip

Archive:  my_archive.zip
   creating: content/model2-0.5b-lora/
  inflating: content/model2-0.5b-lora/adapter_config.json  
  inflating: content/model2-0.5b-lora/adapter_model.safetensors  
  inflating: content/model2-0.5b-lora/README.md  
  inflating: content/model2-0.5b-lora/training_args.bin  
  inflating: content/model2-0.5b-lora/tokenizer_config.json  
   creating: content/model2-0.5b-lora/checkpoint-102/
  inflating: content/model2-0.5b-lora/checkpoint-102/trainer_state.json  
  inflating: content/model2-0.5b-lora/checkpoint-102/adapter_config.json  
  inflating: content/model2-0.5b-lora/checkpoint-102/scheduler.pt  
  inflating: content/model2-0.5b-lora/checkpoint-102/adapter_model.safetensors  
  inflating: content/model2-0.5b-lora/checkpoint-102/rng_state.pth  
  inflating: content/model2-0.5b-lora/checkpoint-102/README.md  
  inflating: content/model2-0.5b-lora/checkpoint-102/training_args.bin  
  inflating: content/model2-0.5b-lora/checkpoint-102/tokenizer_config.json  
  in

In [17]:
!find /content -iname "adapter_config.json"

/content/content/model2-1.5b-lora/checkpoint-102/adapter_config.json
/content/content/model2-1.5b-lora/checkpoint-51/adapter_config.json
/content/content/model2-0.5b-lora/checkpoint-102/adapter_config.json
/content/content/model2-0.5b-lora/checkpoint-51/adapter_config.json
/content/content/model2-0.5b-lora/checkpoint-153/adapter_config.json
/content/content/model2-0.5b-lora/adapter_config.json


In [18]:
!pip install --upgrade torchao
!pip install -q transformers peft accelerate

# --- Cell 6: merge LoRA adapters into base weights (for BOTH models) --------
def merge_and_save(base_model_name, adapter_dir, merged_dir):
    print(f"\nMerging {base_model_name} + {adapter_dir} -> {merged_dir}")
    base = AutoModelForCausalLM.from_pretrained(base_model_name, torch_dtype=torch.bfloat16, device_map="cpu")
    merged = PeftModel.from_pretrained(base, adapter_dir).merge_and_unload()
    merged.save_pretrained(merged_dir)
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
    tokenizer.save_pretrained(merged_dir)
    del base, merged
    gc.collect()
    torch.cuda.empty_cache()


# for cfg in MODEL_CANDIDATES:
#     merged_dir = f"merged_{cfg['tag']}"
#     merge_and_save(cfg["name"], cfg["out_dir"], merged_dir)
merge_and_save("Qwen/Qwen2.5-0.5B-Instruct", "/content/content/model2-0.5b-lora", "merged_0.5b")


Merging Qwen/Qwen2.5-0.5B-Instruct + /content/content/model2-0.5b-lora -> merged_0.5b


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [1]:
# --- Cell 7: convert both merged models to GGUF (Q4_K_M) --------------------
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt

fatal: destination path 'llama.cpp' already exists and is not an empty directory.


In [2]:
!cmake -B llama.cpp/build llama.cpp

-- llama.cpp version: 0.3.0-dev
CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.22.0
-- ggml commit:  d077b4c
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: llama-app
-- Configuring done (1.8s)
-- Generating done (3.1s)
-- Build files have been written to: /content/llama.cpp/build


In [2]:
!cmake --build llama.cpp/build --config Release -j 2

[  2%] Built target vendor-hash
[  3%] Built target ggml-base
[  3%] Building CXX object vendor/cpp-httplib/CMakeFiles/cpp-httplib.dir/httplib.cpp.o
[  3%] Built target llama-common-base
[  4%] Built target llama-ui-embed
[  5%] Built target llama-llava-cli
[  5%] Built target llama-gemma3-cli
[  5%] Built target llama-minicpmv-cli
[  6%] Built target llama-qwen2vl-cli
[  9%] Built target ggml-cpu
[  9%] Provisioning UI assets
-- UI: downloading from b1: https://huggingface.co/buckets/ggml-org/llama-ui/resolve/b1/dist.tar.gz
-- UI: download dist.tar.gz from b1 failed: "HTTP response code said error"
-- UI: downloading from latest: https://huggingface.co/buckets/ggml-org/llama-ui/resolve/latest/dist.tar.gz
-- UI: archive verified and extracted
-- UI: HF download succeeded, stamp updated (latest)
-- UI: gzip compression applied (/content/llama.cpp/build/tools/ui/dist/_gzip)
[  9%] Built target llama-ui-assets
[  9%] Built target ggml
[  9%] Built target llama-ui
[  9%] Building CXX objec

In [6]:
import json

path = "merged_0.5b/tokenizer_config.json"
with open(path) as f:
    cfg = json.load(f)

if isinstance(cfg.get("extra_special_tokens"), list):
    print(f"Found list: {cfg['extra_special_tokens']} — converting to empty dict")
    cfg["extra_special_tokens"] = {}
    with open(path, "w") as f:
        json.dump(cfg, f, indent=2)
    print("Fixed.")
else:
    print(f"extra_special_tokens is already {type(cfg.get('extra_special_tokens'))} — this isn't it, check again.")

Found list: ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>'] — converting to empty dict
Fixed.


In [7]:
!python llama.cpp/convert_hf_to_gguf.py merged_0.5b --outtype f16 --outfile model2_retention_0.5b_f16.gguf

INFO:hf-to-gguf:Loading model: merged_0.5b
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> F16, shape = {896, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {896}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> F16, shape = {4864, 896}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> F16, shape = {896, 4864}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> F16, shape = {896, 4864}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.bfloat16 --> F32, shape = {896}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.bfloat16 --> F32, shape = {128}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.bfloat16 --> F16, shape = {896, 128}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch.bfloat16 -

In [8]:
!./llama.cpp/build/bin/llama-quantize model2_retention_0.5b_f16.gguf model2_retention_0.5b.gguf Q4_K_M

version: 0.3.0-dev (build 1, commit d077b4c)
built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing 'model2_retention_0.5b_f16.gguf' to 'model2_retention_0.5b.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 27 key-value pairs and 290 tensors from model2_retention_0.5b_f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_model_loader: - kv   5:            general.sampling.penalty_repeat f32       

In [9]:
import os
size_mb = os.path.getsize("model2_retention_0.5b.gguf") / (1024 * 1024)
print(f"{size_mb:.1f} MB")

379.4 MB


In [10]:
from google.colab import files
files.download("model2_retention_0.5b.gguf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import json
with open("merged_0.5b/tokenizer_config.json") as f:
    cfg = json.load(f)
print(type(cfg.get("chat_template")))
print(cfg.get("chat_template"))

<class 'NoneType'>
None
